# Text-Only Rulebook RAG with Docling, Qdrant, and Together

This notebook implements a minimal, text-only RAG pipeline for board game rulebooks:

- Docling parses a PDF into per-page text with bounding boxes (for citations).
- FastEmbed creates page-level text embeddings.
- Qdrant stores pages with `game_id`, `page_num`, `text`, and `bboxes`.
- Together-hosted LLM answers questions and returns structured citations:
  - `reasoning`: brief chain-of-thought
  - `answer`: final answer
  - `citations`: list of `{page_num, bbox_indices}` used.

The goal is a **readable, tightly focused** baseline you can tune by:
- swapping the embedding model
- swapping the LLM model
- adjusting the prompt
- changing what payload is sent from Qdrant to the LLM.

In [ ]:
# Stage 0: Config and Imports

from pathlib import Path
from typing import List, Dict, Any
import os
import json
import time
import uuid

os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pydantic import BaseModel
from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

from langchain_together import ChatTogether
from langchain_core.messages import SystemMessage, HumanMessage


# --- Config (tunable) ---

# Either a single PDF  →  Path(".../<game>.pdf")
# Or a folder of PDFs →  Path("../../../rule_book_agent/rulebook_folders/Ark Nova")
RULEBOOK_PATH = Path("../../../rule_book_agent/rulebook_folders/Ark Nova")
GAME_ID = "ark_nova"

QDRANT_PATH = Path("output/qdrant_rulebooks")
COLLECTION_NAME = "rulebook_pages_text_only"

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # swap to tune embeddings
TOGETHER_MODEL_NAME = "openai/gpt-oss-120b"  # swap to tune LLM

# Requires TOGETHER_API_KEY in environment

In [ ]:
# Stage 1: Output Schema (Pydantic)

class Citation(BaseModel):
    doc_name: str        # stem of the source PDF (e.g. "Ark-Nova_342942_rules")
    page_num: int
    bbox_indices: List[int]


class QAWithCitations(BaseModel):
    reasoning: str
    answer: str
    citations: List[Citation]

In [ ]:
# Stage 2: Docling Extraction (per-page text + bboxes)

def _extract_single_pdf(pdf_path: Path, game_id: str, doc_name: str) -> List[Dict[str, Any]]:
    """Parse one PDF with Docling into per-page text and bounding boxes.

    Each page dict contains:
    - game_id
    - doc_name  (PDF stem, e.g. "Ark-Nova_342942_rules")
    - page_num
    - text
    - bboxes: list of {x0, y0, x1, y1, text}
    """
    pipeline_options = PdfPipelineOptions()
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    result = converter.convert(str(pdf_path.resolve()))

    pages_data: List[Dict[str, Any]] = []

    for page_num in sorted(result.document.pages.keys()):
        items_for_page = list(result.document.iterate_items(page_no=page_num))

        text_parts: List[str] = []
        bboxes: List[Dict[str, Any]] = []

        for item, _ in items_for_page:
            item_text = ""
            if getattr(item, "text", None):
                item_text = str(item.text)
                text_parts.append(item_text)

            if getattr(item, "prov", None):
                for prov in item.prov:
                    if getattr(prov, "bbox", None):
                        bbox = prov.bbox
                        bboxes.append(
                            {
                                "x0": bbox.l,
                                "y0": bbox.t,
                                "x1": bbox.r,
                                "y1": bbox.b,
                                "text": item_text,
                            }
                        )

        pages_data.append(
            {
                "game_id": game_id,
                "doc_name": doc_name,
                "page_num": page_num,
                "text": "\n\n".join(text_parts),
                "bboxes": bboxes,
            }
        )

    return pages_data


def extract_pages(source: Path, game_id: str) -> List[Dict[str, Any]]:
    """Extract pages from a single PDF or every PDF in a folder.

    If *source* is a directory every *.pdf file inside it is parsed and each
    page is tagged with its ``doc_name`` (PDF stem).  A single file is treated
    exactly as before.
    """
    source = Path(source)
    if source.is_dir():
        pdf_paths = sorted(source.glob("*.pdf"))
        if not pdf_paths:
            raise ValueError(f"No PDF files found in {source}")
    else:
        pdf_paths = [source]

    all_pages: List[Dict[str, Any]] = []
    for pdf_path in pdf_paths:
        doc_name = pdf_path.stem
        print(f"  Extracting: {pdf_path.name}")
        all_pages.extend(_extract_single_pdf(pdf_path, game_id, doc_name))
    return all_pages

In [ ]:
# Stage 3: Qdrant Indexing (text-only embeddings)

def build_index(pages_data: List[Dict[str, Any]], client: QdrantClient = None):
    if client is None:
        client = QdrantClient(path=str(QDRANT_PATH))
    text_model = TextEmbedding(model_name=EMBED_MODEL_NAME)

    if client.collection_exists(COLLECTION_NAME):
        client.delete_collection(COLLECTION_NAME)
    
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=text_model.embedding_size,
            distance=models.Distance.COSINE,
        ),
    )

    points = []
    for page in pages_data:
        emb = list(text_model.embed([page["text"]]))[0]
        points.append(
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=emb.tolist(),
                payload=page,
            )
        )

    client.upsert(collection_name=COLLECTION_NAME, points=points)
    return client, text_model

In [ ]:
# Stage 4: Text-Only Retrieval (filtered by game_id)

def retrieve_pages(
    client: QdrantClient,
    text_model: TextEmbedding,
    query: str,
    game_id: str,
    k: int = 5,
):
    query_emb = list(text_model.embed([query]))[0]

    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_emb.tolist(),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="game_id",
                    match=models.MatchValue(value=game_id),
                )
            ]
        ),
        limit=k,
        with_payload=True,
    )
    return response.points

In [ ]:
# Stage 5: Together LLM (answer + citations)

def make_llm() -> ChatTogether:
    return ChatTogether(
        model=TOGETHER_MODEL_NAME,
        together_api_key=os.environ["TOGETHER_API_KEY"],
        temperature=0,
    )


def llm_answer_with_citations(query: str, points) -> QAWithCitations:
    pages_for_llm = []
    for p in points:
        payload = p.payload
        bboxes = payload.get("bboxes", [])
        pages_for_llm.append(
            {
                "doc_name": payload["doc_name"],
                "page_num": payload["page_num"],
                "text": payload["text"],
                "bboxes": [
                    {"index": i, "text": b.get("text", "")[:300]}
                    for i, b in enumerate(bboxes)
                    if b.get("text")
                ],
            }
        )

    system_prompt = """You answer board game rulebook questions.

You are given:
- A question.
- Several pages of text from one or more source documents, each page identified
  by its doc_name and page_num, along with an array of text-bounding-boxes (bboxes).

You must:
1. Carefully reason step by step about which page(s) and which bbox texts contain the answer.
2. Return structured output matching the Pydantic schema.

Rules:
- doc_name must exactly match the doc_name field of one of the provided pages.
- page_num must match the page_num of that page.
- bbox_indices must be valid indices into that page's bbox array.
"""

    user_payload = {"question": query, "pages": pages_for_llm}
    user_prompt = json.dumps(user_payload, indent=2)

    llm = make_llm().with_structured_output(QAWithCitations)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]

    return llm.invoke(messages)

In [ ]:
#Extract and index (run once per rulebook/model change)
t0 = time.perf_counter()
pages_data = extract_pages(RULEBOOK_PATH, GAME_ID)
extract_elapsed = time.perf_counter() - t0

existing_client = globals().get('client')
t0 = time.perf_counter()
client, text_model = build_index(pages_data, client=existing_client)
index_elapsed = time.perf_counter() - t0

print(f"Docling extract: {extract_elapsed:.2f}s  |  Build index: {index_elapsed:.2f}s")


In [ ]:
#Ask a question and get structured answer + citations
query = "what defines an animal as a big animal?"

t0 = time.perf_counter()
points = retrieve_pages(client, text_model, query, GAME_ID, k=5)
qdrant_elapsed = time.perf_counter() - t0

t0 = time.perf_counter()
qa = llm_answer_with_citations(query, points)
llm_elapsed = time.perf_counter() - t0

total = extract_elapsed + index_elapsed + qdrant_elapsed + llm_elapsed
print(f"DQdrant retrieval: {qdrant_elapsed:.2f}s  |  LLM: {llm_elapsed:.2f}s")
print(qa.model_dump_json(indent=2))


In [ ]:
# Stage 7: Visualize LLM citations on PDF pages

from collections import defaultdict
from PIL import Image
from IPython.display import display
import fitz


def _resolve_pdf(source: Path, doc_name: str) -> Path:
    """Return the PDF path for *doc_name* given a file or folder *source*."""
    source = Path(source)
    if source.is_dir():
        return source / f"{doc_name}.pdf"
    return source  # single-file mode: ignore doc_name


def show_cited_bboxes(source: Path, points, citations, dpi: int = 150):
    """Render highlighted citations.

    *source* may be a single PDF file (original behaviour) or a folder
    containing multiple PDFs.  Citations are routed to the correct file via
    the ``doc_name`` field that is now part of every Citation and payload.
    """
    # (doc_name, page_num) -> payload
    page_payload_by_key = {
        (p.payload["doc_name"], p.payload["page_num"]): p.payload for p in points
    }

    # group bbox indices by (doc_name, page_num)
    grouped: Dict[tuple, set] = defaultdict(set)
    for citation in citations:
        grouped[(citation.doc_name, citation.page_num)].update(citation.bbox_indices)

    # group pages by doc so we open each PDF only once
    by_doc: Dict[str, Dict[int, set]] = defaultdict(dict)
    for (doc_name, page_num), indices in grouped.items():
        by_doc[doc_name][page_num] = indices

    for doc_name, pages in by_doc.items():
        pdf_path = _resolve_pdf(source, doc_name)
        print(f"\n=== {pdf_path.name} ===")
        fitz_doc = fitz.open(str(pdf_path.resolve()))
        try:
            for page_num in sorted(pages.keys()):
                payload = page_payload_by_key.get((doc_name, page_num))
                if not payload:
                    continue

                page = fitz_doc[page_num - 1]  # PyMuPDF is 0-indexed
                page_height = page.rect.height
                bboxes = payload.get("bboxes", [])

                valid_indices = []
                for idx in sorted(pages[page_num]):
                    if 0 <= idx < len(bboxes):
                        bbox = bboxes[idx]
                        x0, y0 = bbox["x0"], bbox["y0"]
                        x1, y1 = bbox["x1"], bbox["y1"]

                        # Docling coords are BOTTOMLEFT; convert to PyMuPDF TOPLEFT.
                        top_y0 = page_height - y1
                        top_y1 = page_height - y0

                        rect = fitz.Rect(
                            min(x0, x1),
                            min(top_y0, top_y1),
                            max(x0, x1),
                            max(top_y0, top_y1),
                        )

                        annot = page.add_highlight_annot(rect)
                        annot.set_colors(stroke=(1, 1, 0))
                        annot.update()
                        valid_indices.append(idx)

                print(f"  Page {page_num} | highlighted bbox indices: {valid_indices}")
                pix = page.get_pixmap(dpi=dpi)
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                display(img)
        finally:
            fitz_doc.close()


show_cited_bboxes(RULEBOOK_PATH, points, qa.citations)